#### MCP Server

In [1]:
from mcp.server.fastmcp import FastMCP
from pprint import pprint
from typing import Any
import httpx
import random

Globals

In [2]:
NWS_API_BASE = "https://api.weather.gov"
USER_AGENT = "weather-app/1.0"

Server

In [3]:
mcp = FastMCP("Fast MCP")

Server API Call for News Request

In [4]:
async def make_nws_request(url: str) -> dict[str, Any] | None:
    """Make a request to the NWS API with proper error handling."""
    headers = {"User-Agent": USER_AGENT, "Accept": "application/geo+json"}
    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(url, headers=headers, timeout=30.0)
            response.raise_for_status()
            return response.json()
        except Exception:
            return None

Server API Call for Format Alert

In [5]:
def format_alert(feature: dict) -> str:
    """Format an alert feature into a readable string."""
    props = feature["properties"]
    return f"""
Event: {props.get("event", "Unknown")}
Area: {props.get("areaDesc", "Unknown")}
Severity: {props.get("severity", "Unknown")}
Description: {props.get("description", "No description available")}
Instructions: {props.get("instruction", "No specific instructions provided")}
"""

MCP Server Tooling

In [6]:
@mcp.tool()
async def get_alerts(state: str) -> str:
    """Get weather alerts for a US state.

    Args:
        state: Two-letter US state code (e.g. CA, NY)
    """
    url = f"{NWS_API_BASE}/alerts/active/area/{state}"
    data = await make_nws_request(url)

    if not data or "features" not in data:
        return "Unable to fetch alerts or no alerts found."

    if not data["features"]:
        return "No active alerts for this state."

    alerts = [format_alert(feature) for feature in data["features"]]
    return "\n---\n".join(alerts)

In [7]:
@mcp.tool()
async def get_forecast(latitude: float, longitude: float) -> str:
    """Get weather forecast for a location.

    Args:
        latitude: Latitude of the location
        longitude: Longitude of the location
    """
    # First get the forecast grid endpoint
    points_url = f"{NWS_API_BASE}/points/{latitude},{longitude}"
    points_data = await make_nws_request(points_url)

    if not points_data:
        return "Unable to fetch forecast data for this location."

    # Get the forecast URL from the points response
    forecast_url = points_data["properties"]["forecast"]
    forecast_data = await make_nws_request(forecast_url)

    if not forecast_data:
        return "Unable to fetch detailed forecast."

    # Format the periods into a readable forecast
    periods = forecast_data["properties"]["periods"]
    forecasts = []
    for period in periods[:5]:  # Only show next 5 periods
        forecast = f"""
            {period["name"]}:
            Temperature: {period["temperature"]}°{period["temperatureUnit"]}
            Wind: {period["windSpeed"]} {period["windDirection"]}
            Forecast: {period["detailedForecast"]}
        """
        forecasts.append(forecast)

    return "\n---\n".join(forecasts)

MCP Server Prompt

In [8]:
@mcp.prompt()
def prompt():
    """Weather Assistant the fetches weather"""
    
    
    return """
    You are a helpful assistant that answers user questions about the weather.

    You can use the following tools/resources to answer user questions:
    - get_forcast: Get weather forcast for a certain location
    - get_alerts: Get weather alerts for a US state

    If the user asks a question that is not related to the weather, you should say "I'm sorry, I can only answer questions about LangChain, LangGraph and LangSmith."

    You may try multiple tool and resource calls to answer the user's question.

    You may also ask clarifying questions to the user to better understand their question.
    """

Starting the Server

In [9]:
await mcp.run_sse_async()

INFO:     Started server process [49757]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:57632 - "GET /sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:57633 - "POST /messages/?session_id=bedb47a2a49e4a2883f571c6388fe1df HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57633 - "POST /messages/?session_id=bedb47a2a49e4a2883f571c6388fe1df HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57633 - "POST /messages/?session_id=bedb47a2a49e4a2883f571c6388fe1df HTTP/1.1" 202 Accepted


[03/11/26 11:17:00] INFO     Processing request of type GetPromptRequest                              ]8;id=990285;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=379515;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#720\720]8;;\

INFO:     127.0.0.1:57634 - "GET /sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:57635 - "POST /messages/?session_id=928c5a5ef2084c829efa1d748bf590c4 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57635 - "POST /messages/?session_id=928c5a5ef2084c829efa1d748bf590c4 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57635 - "POST /messages/?session_id=928c5a5ef2084c829efa1d748bf590c4 HTTP/1.1" 202 Accepted


                    INFO     Processing request of type ListResourcesRequest                          ]8;id=31278;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=90243;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#720\720]8;;\

...

                    INFO     HTTP Request: GET https://api.weather.gov/points/47.6062,-122.3321     ]8;id=878819;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=605826;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: GET                                                      ]8;id=133141;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=727528;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             https://api.weather.gov/gridpoints/SEW/125,68/forecast "HTTP/1.1 200                  
                             OK"                                                                                   

INFO:     127.0.0.1:58472 - "POST /messages/?session_id=34918b3d518f4e28bc0cad1e82a66f6c HTTP/1.1" 202 Accepted


                    INFO     Processing request of type ListToolsRequest                              ]8;id=475940;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=816503;file:///Users/matmtz/Documents/repos/lang-chain/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#720\720]8;;\

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [49757]
